# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cokezero20/FlyRank_AI_ML_Internship_NATIVIDAD/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding #4 — The Freshness Multiplier

**The claim:** Content older than 365 days that was refreshed within 30 days showed a 3.2x health boost (10.7 to 34.5) and 57x more impressions (71 to 4,039).

**Claim type:** Descriptive, presented as observational — but the "57x" framing reads as a lever, which edges toward causal.

**Where does the label come from?** The label is the freshness bucket — days since last content update — drawn from content metadata. Pages are grouped into freshness tiers (0-30d, 31-90d, 91-180d, 181-360d, 361+), and performance is compared across tiers within the 365+ age band. The label is not derived from the outcome, which is good. However, the freshness timestamp and the 90-day impression window may overlap: if a page was refreshed 10 days ago, both the "0-30d fresh" label and the impression count share the same recent window, which inflates the apparent boost.

**Does the validation design carry the claim?** Not fully. The comparison is a direct aggregate — old-refreshed vs old-stale — with no matched control group. Pages that got refreshed were selected by someone who already believed they were worth updating. That selection effect could explain much of the 57x gap. A stronger design would compare refreshed pages against similar-visibility pages that were NOT refreshed, or use a before/after design on the same pages. The paper's own language ("refresh timing is one of the strongest measured levers") is slightly stronger than what this uncontrolled comparison supports. A safer phrasing would be: "refreshed old pages were observed to have materially higher impressions than unrefreshed old pages in this snapshot."

---

### ML Appendix — Feature Importance (Health Score Prediction)

**The claim:** Average position is the #1 predictor of health score at 43% importance, followed by impressions at 32%.

**Claim type:** Descriptive — the paper explicitly notes this is "model behavior, not a standalone optimization order."

**Where does the label come from?** Health score is computed as impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts). The label is directly constructed from four of the input features. This means a model trained to predict health score will find position and impressions important by definition — they are 60% of the formula. This is label-derived leakage by construction.

**Does the validation design carry the claim?** The 80/20 holdout split is technically sound for testing prediction accuracy, but that's not the issue. The problem is the target definition: no amount of honest splitting fixes the fact that predicting a composite built from its own inputs will always produce circular importance rankings. The paper acknowledges this clearly ("high importance is therefore expected and does not imply external causation"), which is good practice. The constructive next step would be to retrain on an external outcome — impression growth, click change, or decline — that is not built from the input features. The paper's own direct aggregate comparisons (Findings 1–8) are more trustworthy than this appendix precisely because they don't route through a composite target.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Week-5 split:** 80/20 random stratified split. Content from the same client could appear in both train and test, letting the model memorize client-specific patterns.

**Honest split:** GroupKFold by `client_hash_id`. All content from one client stays together — either fully in train or fully in test. This asks: "does the model work on a client it has never seen?"

**What the gap means:** If performance drops significantly under the grouped split, the Week-5 score was partly borrowed from client memorization, not from learning real decline signals.

In [1]:
import pandas as pd
import numpy as np
from datetime import date
from datasets import load_dataset
from google.colab import userdata
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
HF_TOKEN = userdata.get('HF_Token')

# ── Load data ────────────────────────────────────────────────────────
dataset = load_dataset(
    'FlyRank/internship-warehouse',
    name='fact_content_daily_performance',
    token=HF_TOKEN,
    streaming=True
)

print("Loading January–April 2026 data...")
data_rows = []
for batch in dataset['train'].iter(batch_size=100000):
    batch_df = pd.DataFrame(batch)
    if isinstance(batch_df['report_date'].iloc[0], str):
        batch_df['report_date'] = pd.to_datetime(batch_df['report_date']).dt.date
    data_batch = batch_df[
        (batch_df['report_date'] >= date(2026, 1, 1)) &
        (batch_df['report_date'] <= date(2026, 4, 30)) &
        (batch_df['ga4_data_available'] == True)
    ]
    if len(data_batch) > 0:
        data_rows.append(data_batch)

df = pd.concat(data_rows, ignore_index=True)
df['month'] = pd.to_datetime(df['report_date']).dt.month
print(f"Total rows: {len(df):,}")

# ── Create label ─────────────────────────────────────────────────────
monthly_impr = (
    df[df['month'].isin([3, 4])]
    .groupby(['content_hash_id', 'month'])['gsc_impressions']
    .sum()
    .unstack(fill_value=0)
)
monthly_impr.columns = ['mar_impressions', 'apr_impressions']
monthly_impr['is_declining_label'] = (
    monthly_impr['apr_impressions'] < (0.8 * monthly_impr['mar_impressions'])
).astype(int)

# ── Build features from Jan–Mar ──────────────────────────────────────
df_features = df[df['month'].isin([1, 2, 3])]

feature_list = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position',
                'ga4_pageviews', 'ga4_sessions', 'ga4_users',
                'ga4_engaged_sessions', 'ga4_total_engagement_sec',
                'sessions_organic', 'sessions_direct', 'sessions_referral',
                'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events']

content = df_features.groupby('content_hash_id')[feature_list].agg(['sum', 'mean']).reset_index()
content.columns = ['content_hash_id'] + [f"{f}_{agg}" for f, agg in content.columns[1:]]
content['ctr'] = content['gsc_clicks_sum'] / content['gsc_impressions_sum'].replace(0, np.nan)
content['ctr'] = content['ctr'].fillna(0)

# Get client mapping
client_map = df_features.groupby('content_hash_id')['client_hash_id'].first().reset_index()
content = content.merge(client_map, on='content_hash_id', how='left')
content = content.merge(monthly_impr[['is_declining_label']], on='content_hash_id', how='inner')
content = content.dropna()

feature_cols = [c for c in content.columns if c not in ['content_hash_id', 'is_declining_label', 'client_hash_id']]
X = content[feature_cols]
y = content['is_declining_label']
groups = content['client_hash_id']

# ── Helper ────────────────────────────────────────────────────────────
def precision_at_k(y_true, y_scores, k):
    order = np.argsort(-np.asarray(y_scores))
    return np.asarray(y_true)[order[:k]].mean()

# ── BEFORE: Random split (same as Week-5) ─────────────────────────────
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

rf_random = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
rf_random.fit(X_train_r, y_train_r)
rf_random_probs = rf_random.predict_proba(X_test_r)[:, 1]
rf_random_preds = rf_random.predict(X_test_r)

# ── AFTER: Grouped split by client ────────────────────────────────────
gkf = GroupKFold(n_splits=5)
# Use the last fold as our train/test to get a single comparison
for train_idx, test_idx in gkf.split(X, y, groups):
    pass  # take the last fold

X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
rf_grouped.fit(X_train_g, y_train_g)
rf_grouped_probs = rf_grouped.predict_proba(X_test_g)[:, 1]
rf_grouped_preds = rf_grouped.predict(X_test_g)

# ── COMPARISON TABLE ──────────────────────────────────────────────────
print("=" * 65)
print("BEFORE vs AFTER: Random Split vs Grouped Split (by client)")
print("=" * 65)
print(f"{'Metric':<20} {'Random Split':>15} {'Grouped Split':>15} {'Gap':>10}")
print("-" * 65)

for k in [10, 20, 50, 100]:
    r = precision_at_k(y_test_r.values, rf_random_probs, k)
    g = precision_at_k(y_test_g.values, rf_grouped_probs, k)
    print(f"{'Precision@'+str(k):<20} {r:>15.3f} {g:>15.3f} {g-r:>+10.3f}")

r_f1 = f1_score(y_test_r, rf_random_preds)
g_f1 = f1_score(y_test_g, rf_grouped_preds)
r_auc = roc_auc_score(y_test_r, rf_random_probs)
g_auc = roc_auc_score(y_test_g, rf_grouped_probs)

print(f"{'F1':<20} {r_f1:>15.3f} {g_f1:>15.3f} {g_f1-r_f1:>+10.3f}")
print(f"{'AUC':<20} {r_auc:>15.3f} {g_auc:>15.3f} {g_auc-r_auc:>+10.3f}")

r_base = y_test_r.mean()
g_base = y_test_g.mean()
print(f"\nBase rate (random):  {r_base:.3f}")
print(f"Base rate (grouped): {g_base:.3f}")
print(f"Random test size:    {len(y_test_r):,}")
print(f"Grouped test size:   {len(y_test_g):,}")
print(f"Unique clients in grouped test: {content.iloc[test_idx]['client_hash_id'].nunique()}")


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading January–April 2026 data...
Total rows: 1,199,364
BEFORE vs AFTER: Random Split vs Grouped Split (by client)
Metric                  Random Split   Grouped Split        Gap
-----------------------------------------------------------------
Precision@10                   0.800           0.700     -0.100
Precision@20                   0.900           0.700     -0.200
Precision@50                   0.920           0.620     -0.300
Precision@100                  0.960           0.630     -0.330
F1                             0.723           0.637     -0.086
AUC                            0.754           0.549     -0.205

Base rate (random):  0.530
Base rate (grouped): 0.557
Random test size:    13,386
Grouped test size:   13,384
Unique clients in grouped test: 8


### Interpretation

**The gap is real and meaningful.** When the model can't see content from the same client during training, every metric drops:

| Metric | Random Split | Grouped Split | Gap |
|--------|-------------|---------------|-----|
| Precision@100 | 0.960 | 0.630 | -0.330 |
| AUC | 0.754 | 0.549 | -0.205 |
| F1 | 0.723 | 0.637 | -0.086 |

**What this means:** The Week-5 model was borrowing roughly a third of its precision@100 performance from client memorization. Content from the same client shares patterns — similar word counts, similar publishing cadence, similar position profiles — and the random split let the model learn "this client's content declines" rather than "content with these signals declines."

**AUC dropped to 0.549** — barely above the 0.50 coin-flip line. This means the model's ability to rank declining content above non-declining content nearly disappears on unseen clients.

**The honest number is the grouped number.** Precision@100 of 0.630 against a base rate of 0.557 is a small but real improvement. The model still knows something, but it knows much less than the Week-5 score suggested.

**Note:** Only 8 unique clients landed in the grouped test fold. With so few clients, a single unusual client can shift the metrics substantially. This is a limitation of having only 57 brands in the dataset — grouped validation is honest but noisy at this scale.

**Limitation:** This reports a single fold, not a 5-fold average. With only 8 clients in the test fold, results are sensitive to which clients land in test. A cross-validated average across all 5 folds would be more stable but harder to compare directly against the Week-5 table.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


The same attack checklist from the hunting-leakage-and-validating skill, applied to the final Week-5 feature set.

In [2]:
# ── Leakage Audit: Attack Checklist ───────────────────────────────────

print("=" * 65)
print("LEAKAGE AUDIT — ATTACK CHECKLIST")
print("=" * 65)

# CHECK 1: Timeline — all features strictly before label window?
print("\n[1] TIMELINE CHECK")
print("    Label: March vs April impressions (is_declining_label)")
print("    Features built from: January–March daily data only")
print("    April data in features? NO")
print("    ✓ Features are strictly before the label window")

# CHECK 2: Label-derived or sibling columns?
print("\n[2] LABEL-DERIVED FEATURE CHECK")
print("    Label uses: gsc_impressions (March sum vs April sum)")
print("    Features that include gsc_impressions:")

# Identify suspect features
suspects = [c for c in feature_cols if 'gsc_impressions' in c]
print(f"    → {suspects}")
print("    These are SIBLING features — they measure the same metric")
print("    (impressions) in the feature window that the label measures")
print("    in the label window. Not direct leakage, but suspect.")

# CHECK 2b: Train-without test on suspect features
print("\n    Running train-without test on gsc_impressions features...")

clean_features = [c for c in feature_cols if 'gsc_impressions' not in c]

# With suspect
rf_with = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
rf_with.fit(X_train_g, y_train_g)
auc_with = roc_auc_score(y_test_g, rf_with.predict_proba(X_test_g)[:, 1])

# Without suspect
rf_without = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
rf_without.fit(X_train_g[clean_features], y_train_g)
auc_without = roc_auc_score(y_test_g, rf_without.predict_proba(X_test_g[clean_features])[:, 1])

print(f"    AUC WITH gsc_impressions features:    {auc_with:.3f}")
print(f"    AUC WITHOUT gsc_impressions features: {auc_without:.3f}")
print(f"    Drop: {auc_without - auc_with:+.3f}")

if auc_with - auc_without > 0.15:
    print("    ⚠ LARGE DROP — gsc_impressions may be doing most of the work")
elif auc_with - auc_without > 0.05:
    print("    ⚠ MODERATE DROP — gsc_impressions contributes meaningfully")
else:
    print("    ✓ SMALL DROP — gsc_impressions is not dominating the model")

# CHECK 3: Product flags or decision-derived features?
print("\n[3] PRODUCT FLAG CHECK")
print("    Features used:")
for f in sorted(feature_cols):
    print(f"      {f}")
print("\n    Any FlyRank flags, scores, or optimization labels? NO")
print("    Any excluded columns (gsc_data_available, ga4_data_available, gsc_sum_position)? NO")
print("    ✓ No decision-derived features in the model")

# CHECK 4: Top feature sanity check
print("\n[4] TOP FEATURE SANITY CHECK")
from sklearn.inspection import permutation_importance
perm = permutation_importance(rf_grouped, X_test_g, y_test_g, n_repeats=10, random_state=SEED, n_jobs=-1)
imp_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': perm.importances_mean
}).sort_values('importance', ascending=False)

print("    Top 5 features (grouped split, permutation importance):")
for _, row in imp_df.head(5).iterrows():
    print(f"      {row['feature']:<40} {row['importance']:.4f}")

top_feat = imp_df.iloc[0]
if top_feat['importance'] > 0.10:
    print(f"\n    ⚠ Top feature '{top_feat['feature']}' looks suspiciously dominant")
    print("      Investigate whether it is a proxy for the label")
else:
    print(f"\n    ✓ No single feature dominates suspiciously")

# CHECK 5: Base rate printed?
print(f"\n[5] BASE RATE")
print(f"    Grouped test base rate: {y_test_g.mean():.3f}")
print(f"    ✓ Printed alongside all metrics")

# CHECK 6: In-sample vs out-of-sample
print(f"\n[6] IN-SAMPLE vs OUT-OF-SAMPLE")
train_auc = roc_auc_score(y_train_g, rf_grouped.predict_proba(X_train_g)[:, 1])
test_auc = roc_auc_score(y_test_g, rf_grouped.predict_proba(X_test_g)[:, 1])
print(f"    Train AUC: {train_auc:.3f}")
print(f"    Test AUC:  {test_auc:.3f}")
print(f"    Gap:       {train_auc - test_auc:+.3f}")
if train_auc - test_auc > 0.20:
    print("    ⚠ LARGE GAP — model is overfitting")
elif train_auc - test_auc > 0.10:
    print("    ⚠ MODERATE GAP — some overfitting present")
else:
    print("    ✓ Gap is within normal range")

LEAKAGE AUDIT — ATTACK CHECKLIST

[1] TIMELINE CHECK
    Label: March vs April impressions (is_declining_label)
    Features built from: January–March daily data only
    April data in features? NO
    ✓ Features are strictly before the label window

[2] LABEL-DERIVED FEATURE CHECK
    Label uses: gsc_impressions (March sum vs April sum)
    Features that include gsc_impressions:
    → ['gsc_impressions_sum', 'gsc_impressions_mean']
    These are SIBLING features — they measure the same metric
    (impressions) in the feature window that the label measures
    in the label window. Not direct leakage, but suspect.

    Running train-without test on gsc_impressions features...
    AUC WITH gsc_impressions features:    0.549
    AUC WITHOUT gsc_impressions features: 0.537
    Drop: -0.012
    ✓ SMALL DROP — gsc_impressions is not dominating the model

[3] PRODUCT FLAG CHECK
    Features used:
      ctr
      ga4_engaged_sessions_mean
      ga4_engaged_sessions_sum
      ga4_pageviews_mean

### Leakage Audit Summary

| Check | Result | Verdict |
|-------|--------|---------|
| Timeline | Features use Jan–Mar only, label uses Mar vs Apr | ✓ Clean |
| Label-derived features | gsc_impressions is in both features and label formula, but removing it only drops AUC by 0.012 | ✓ Sibling, not leaking |
| Product flags | No FlyRank flags, scores, or optimization labels in features | ✓ Clean |
| Top feature dominance | gsc_impressions_mean leads at 0.0226 — meaningful but not suspiciously perfect | ✓ Clean |
| Base rate | Printed alongside all metrics | ✓ Clean |
| In-sample vs out-of-sample | Train AUC 0.832 vs test AUC 0.549 — gap of 0.283 | ⚠ Overfitting |

### Key finding: overfitting, not leakage

There is no data leakage in the feature set. The timeline is clean, no product flags are used, and the sibling feature (gsc_impressions) is not dominating the model.

The real problem is overfitting. The model achieves 0.832 AUC on training data but only 0.549 on unseen clients — a 0.283 gap. This means the Random Forest is memorizing training patterns rather than learning generalizable decline signals. Combined with the Section 2 finding (only 8 clients in the test fold), this suggests the model is learning client-specific behavior that doesn't transfer.

### What this means for claims

Any claim about model performance must use the grouped, out-of-sample number (AUC 0.549), not the random split number (AUC 0.754) or the training number (AUC 0.832). The honest statement is: the model provides marginal separation above the base rate when evaluated on unseen clients, and overfits substantially on known clients.

In [3]:
# ── ADDITION 2: Train-without test on RANDOM split (cleaner signal) ──

print("=" * 65)
print("LEAKAGE RETEST — Train-without on Random Split")
print("=" * 65)

# With gsc_impressions (random split)
rf_r_with = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
rf_r_with.fit(X_train_r, y_train_r)
auc_r_with = roc_auc_score(y_test_r, rf_r_with.predict_proba(X_test_r)[:, 1])

# Without gsc_impressions (random split)
rf_r_without = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
rf_r_without.fit(X_train_r[clean_features], y_train_r)
auc_r_without = roc_auc_score(y_test_r, rf_r_without.predict_proba(X_test_r[clean_features])[:, 1])

print(f"  AUC WITH gsc_impressions (random split):    {auc_r_with:.3f}")
print(f"  AUC WITHOUT gsc_impressions (random split): {auc_r_without:.3f}")
print(f"  Drop: {auc_r_without - auc_r_with:+.3f}")

if auc_r_with - auc_r_without > 0.15:
    print("  ⚠ LARGE DROP — gsc_impressions carries significant weight on random split")
elif auc_r_with - auc_r_without > 0.05:
    print("  ⚠ MODERATE DROP — gsc_impressions contributes meaningfully on random split")
else:
    print("  ✓ SMALL DROP — gsc_impressions is not dominating even on random split")

# ── ADDITION 3: Error examples from grouped split ────────────────────

print("\n" + "=" * 65)
print("ERROR EXAMPLES — Grouped Split (unseen clients)")
print("=" * 65)

grouped_results = content.iloc[test_idx][['content_hash_id', 'client_hash_id']].copy()
grouped_results['y_true'] = y_test_g.values
grouped_results['y_pred'] = rf_grouped_preds
grouped_results['y_prob'] = rf_grouped_probs

# Merge features for inspection
grouped_errors = grouped_results[grouped_results['y_pred'] != grouped_results['y_true']].copy()
grouped_errors = grouped_errors.merge(
    content[['content_hash_id'] + feature_cols],
    on='content_hash_id'
)

# 3 false positives
print("\n3 FALSE POSITIVES — predicted decline, actually stable (grouped split)")
print("-" * 65)
worst_fp = grouped_errors[grouped_errors['y_true'] == 0].nlargest(3, 'y_prob')
for _, row in worst_fp.iterrows():
    print(f"\n  {row['content_hash_id']}  (client: {row['client_hash_id'][:20]})")
    print(f"    Prob: {row['y_prob']:.3f}  |  Actual: not declining")
    print(f"    Impressions (sum): {row['gsc_impressions_sum']:.0f}  |  Avg position: {row['gsc_avg_position_mean']:.1f}  |  CTR: {row['ctr']:.4f}")

# 3 false negatives
print("\n3 FALSE NEGATIVES — missed real declines (grouped split)")
print("-" * 65)
worst_fn = grouped_errors[grouped_errors['y_true'] == 1].nsmallest(3, 'y_prob')
for _, row in worst_fn.iterrows():
    print(f"\n  {row['content_hash_id']}  (client: {row['client_hash_id'][:20]})")
    print(f"    Prob: {row['y_prob']:.3f}  |  Actual: declining")
    print(f"    Impressions (sum): {row['gsc_impressions_sum']:.0f}  |  Avg position: {row['gsc_avg_position_mean']:.1f}  |  CTR: {row['ctr']:.4f}")

LEAKAGE RETEST — Train-without on Random Split
  AUC WITH gsc_impressions (random split):    0.754
  AUC WITHOUT gsc_impressions (random split): 0.743
  Drop: -0.010
  ✓ SMALL DROP — gsc_impressions is not dominating even on random split

ERROR EXAMPLES — Grouped Split (unseen clients)

3 FALSE POSITIVES — predicted decline, actually stable (grouped split)
-----------------------------------------------------------------

  content_535b63c41ddc2657  (client: client_2094c6eb08031)
    Prob: 0.867  |  Actual: not declining
    Impressions (sum): 27  |  Avg position: 6.3  |  CTR: 0.0000

  content_36f3bea63c4e07b5  (client: client_2094c6eb08031)
    Prob: 0.862  |  Actual: not declining
    Impressions (sum): 412  |  Avg position: 9.5  |  CTR: 0.0000

  content_b9d9335a73c80e4f  (client: client_2094c6eb08031)
    Prob: 0.853  |  Actual: not declining
    Impressions (sum): 250  |  Avg position: 11.8  |  CTR: 0.0040

3 FALSE NEGATIVES — missed real declines (grouped split)
----------------

### Addition 2 — Leakage Retest Verdict

AUC dropped only 0.010 (0.754 → 0.743) when gsc_impressions features were removed on the random split. This confirms that gsc_impressions is not leaking — even when the model has enough signal to work with, removing the sibling feature barely changes performance. The model relies on a spread of features, not on a single column that mirrors the label.

### Addition 3 — Error Examples from Grouped Split

**False positives — all from the same client** (`client_2094c6eb08031`). Low impressions (27–412), zero or near-zero CTR, decent positions (6–12). The model saw similar profiles from other clients during training and learned "this pattern declines." But this client's content is stable at low activity. The model has never seen this client before and applies patterns that don't transfer.

**False negatives — also all from one client** (`client_a80fca3f171ed`). High impressions (1,011–1,894), strong positions (2–5), nonzero CTR. The model sees "healthy content" and predicts stable — but these pages declined anyway. On unseen clients, the model cannot detect decline that comes from client-specific or external factors invisible in the feature set.

**The pattern:** Both error types cluster by client. This reinforces the Section 2 finding — the model's performance is client-dependent. On clients it has trained on, it works. On new clients, it applies learned stereotypes that may not hold. This is memorization, not generalization.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Identifying my boldest claims from Week-5

**Original claim 1 (Week-5 comparison table):**
"Random Forest earns its complexity. It beats Logistic Regression at every metric and substantially beats the baseline at precision@K. The improvement is consistent, not concentrated at one K value — this is a real gain, not noise."

**Rewritten (safe language):**
"On a random stratified split, the Random Forest model was observed to outperform both Logistic Regression and the rule baseline across precision@K, F1, and AUC. However, under a grouped split by client, AUC dropped from 0.754 to 0.549, indicating that a substantial portion of the measured improvement came from client-level memorization rather than generalizable decline signals. The model provides directional value for prioritizing content review but should not be treated as a reliable predictor on unseen clients without further validation."

---

**Original claim 2 (Week-5 error analysis):**
"The model's blind spots mirror the baseline's but are sharper. It struggles at both extremes: content too small to read a trend from (false positives), and content that looks healthy but declines from external causes the features can't capture (false negatives)."

**Rewritten (safe language):**
"In the reviewed error cases, the model's false positives were observed among low-impression content that appeared stable rather than declining, and its false negatives appeared among high-visibility content where decline may have been driven by factors not captured in the feature set. These patterns are directional observations from a small number of reviewed cases, not a complete characterization of model failure modes."

---

**Original claim 3 (Week-5 interpretation):**
"gsc_impressions_mean is the strongest signal. Content losing visibility day-to-day is the most direct indicator of decline. Makes sense: declining content shows up less."

**Rewritten (safe language):**
"In this dataset, gsc_impressions_mean was measured as the top-ranked feature by permutation importance under both random and grouped splits. This is consistent with the expectation that content showing reduced daily impressions is more likely to meet the decline threshold. However, because the label is defined by impression change (March vs April), this feature is a sibling of the outcome — it measures the same underlying metric in an earlier window. Its importance is plausible but should be interpreted as a supporting signal for decision-making, not as independent causal evidence that impression level drives decline."

---

### Language rules applied

| Unsafe | Safe replacement |
|--------|-----------------|
| "this is a real gain" | "was observed to outperform" |
| "the most direct indicator" | "was measured as the top-ranked feature" |
| "declining content shows up less" | "consistent with the expectation that" |
| "earns its complexity" | "under grouped validation, the gap narrows substantially" |
| "blind spots are sharper" | "directional observations from reviewed cases" |

### Summary

The Week-5 model provides directional decision-support value for prioritizing content review. It is not a reliable standalone predictor of decline on unseen clients. The honest performance number is AUC 0.549 on a grouped split — marginally above the 0.50 baseline, with substantial overfitting observed. All claims in future work will use grouped, out-of-sample metrics as the primary evidence.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.